In [2]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master("local[*]")
         .appName("monografia")
         .config("spark.driver.memory", "4g")
         .getOrCreate())

# Prueba 1: Spark arranca y cuenta
print("Prueba 1:", spark.range(10).count())

# Prueba 2: Spark guarda y lee Parquet (aquí se nota si winutils está mal)
spark.range(10).write.mode("overwrite").parquet("../data/prueba.parquet")
print("Prueba 2:", spark.read.parquet("../data/prueba.parquet").count())

# Prueba 3: Spark lanza procesos de Python (lo necesitan las variantes RDD y UDF)
print("Prueba 3:", spark.sparkContext.parallelize([1, 2, 3]).map(lambda x: x * 2).collect())

Prueba 1: 10
Prueba 2: 10
Prueba 3: [2, 4, 6]


In [3]:
import platform, os, sys, psutil

info = f"""Sistema operativo: {platform.platform()}
Procesador: {platform.processor()}
Núcleos lógicos: {os.cpu_count()}
Memoria RAM: {psutil.virtual_memory().total / 1e9:.1f} GB
Python: {sys.version.split()[0]}
Java: {spark.sparkContext._jvm.System.getProperty("java.version")}
Spark: {spark.version}"""

print(info)
with open("../results/equipo.txt", "w", encoding="utf-8") as f:
    f.write(info)

Sistema operativo: Windows-10-10.0.26200-SP0
Procesador: AMD64 Family 23 Model 96 Stepping 1, AuthenticAMD
Núcleos lógicos: 16
Memoria RAM: 16.6 GB
Python: 3.11.9
Java: 17.0.20.1
Spark: 3.5.9


In [1]:
import requests, time
from pathlib import Path

URL = "https://www.datos.gov.co/resource/jbjy-vk9h.csv"
COLUMNAS = ("id_contrato,departamento,nombre_entidad,modalidad_de_contratacion,"
            "proveedor_adjudicado,fecha_de_firma,valor_del_contrato")
LOTE = 500_000                       # filas por parte
destino = Path("../data/raw")
destino.mkdir(parents=True, exist_ok=True)

def descargar_anio(anio):
    parte, offset, total = 1, 0, 0
    while True:
        archivo = destino / f"secop_{anio}_parte{parte:02d}.csv"

        if archivo.exists():                         # ya se bajó antes: no repetir
            filas = sum(1 for _ in open(archivo, encoding="utf-8")) - 1
        else:
            params = {
                "$select": COLUMNAS,
                "$where": f"fecha_de_firma between '{anio}-01-01T00:00:00' and '{anio}-12-31T23:59:59'",
                "$order": ":id",
                "$limit": LOTE,
                "$offset": offset,
            }
            for intento in range(1, 6):              # hasta 5 intentos
                try:
                    r = requests.get(URL, params=params, timeout=900)
                    r.raise_for_status()
                    break
                except Exception as e:
                    print(f"  {anio} parte {parte}: intento {intento} falló ({e}); reintento en 30 s")
                    time.sleep(30)
            else:
                raise RuntimeError(f"No se pudo descargar {anio} parte {parte}")
            archivo.write_bytes(r.content)
            filas = r.content.count(b"\n") - 1

        total += filas
        print(f"{anio} parte {parte}: {filas:,} filas")
        if filas < LOTE:                             # última parte del año
            return total
        parte += 1
        offset += LOTE

for anio in [2023, 2024, 2025]:
    t0 = time.time()
    n = descargar_anio(anio)
    print(f"==> {anio}: {n:,} filas en {(time.time() - t0) / 60:.1f} min\n")

2023 parte 1: 500,000 filas
2023 parte 2: 335,462 filas
==> 2023: 835,462 filas en 1.1 min

2024 parte 1: 500,000 filas
2024 parte 2: 381,978 filas
==> 2024: 881,978 filas en 1.8 min

2025 parte 1: 500,000 filas
2025 parte 2: 500,000 filas
2025 parte 3: 17,075 filas
==> 2025: 1,017,075 filas en 1.4 min



In [2]:
from pyspark.sql import SparkSession, functions as F
from pathlib import Path

spark = (SparkSession.builder
         .master("local[*]")
         .appName("monografia")
         .config("spark.driver.memory", "8g")
         .getOrCreate())

# 1. Leer todos los CSV descargados
crudo = spark.read.option("header", True).csv("../data/raw/*.csv")
n_crudo = crudo.count()

# 2. Convertir tipos: valor a número y fecha a tipo fecha
df = (crudo
      .withColumn("valor", F.col("valor_del_contrato").cast("double"))
      .withColumn("fecha", F.to_date(F.substring("fecha_de_firma", 1, 10)))
      .withColumn("anio", F.year("fecha"))
      .select("id_contrato", "departamento", "nombre_entidad",
              "modalidad_de_contratacion", "proveedor_adjudicado",
              "fecha", "anio", "valor"))

# 3. Quitar filas sin valor o sin fecha
df_validos = df.dropna(subset=["valor", "fecha"])
n_validos = df_validos.count()

# 4. Quitar contratos repetidos
df_final = df_validos.dropDuplicates(["id_contrato"])
n_final = df_final.count()

# 5. Guardar en Parquet
df_final.write.mode("overwrite").parquet("../data/secop.parquet")

# 6. Tamaños en disco
def tamano_mb(patron):
    return sum(p.stat().st_size for p in Path("../data").glob(patron) if p.is_file()) / 1e6

resumen = f"""Filas descargadas:            {n_crudo:,}
Sin valor o sin fecha (fuera): {n_crudo - n_validos:,}
Duplicados (fuera):            {n_validos - n_final:,}
Filas finales:                 {n_final:,}
Columnas finales:              {len(df_final.columns)}
Tamaño CSV:                    {tamano_mb('raw/*.csv'):,.1f} MB
Tamaño Parquet:                {tamano_mb('secop.parquet/*.parquet'):,.1f} MB"""

print(resumen)
with open("../results/tabla1_datos.txt", "w", encoding="utf-8") as f:
    f.write(resumen)

Filas descargadas:            2,734,515
Sin valor o sin fecha (fuera): 0
Duplicados (fuera):            223
Filas finales:                 2,734,292
Columnas finales:              8
Tamaño CSV:                    473.0 MB
Tamaño Parquet:                94.6 MB


In [3]:
d = spark.read.parquet("../data/secop.parquet")

print("Contratos por año:")
d.groupBy("anio").count().orderBy("anio").show()

print("Resumen del valor del contrato (pesos):")
d.select(
    F.count(F.when(F.col("valor") <= 0, 1)).alias("valor_cero_o_negativo"),
    F.min("valor").alias("minimo"),
    F.expr("percentile_approx(valor, 0.5)").alias("mediana"),
    F.max("valor").alias("maximo"),
).show(truncate=False)

print("Departamentos distintos:", d.select("departamento").distinct().count())

Contratos por año:
+----+-------+
|anio|  count|
+----+-------+
|2023| 835394|
|2024| 881919|
|2025|1016979|
+----+-------+

Resumen del valor del contrato (pesos):
+---------------------+------+----------+----------+
|valor_cero_o_negativo|minimo|mediana   |maximo    |
+---------------------+------+----------+----------+
|19543                |0.0   |1.926054E7|6.45384E15|
+---------------------+------+----------+----------+

Departamentos distintos: 34
